# 🛡️ Notebook 03 — Data Quality Framework

**Goal:** Build a reusable data quality validation framework. Quarantine bad records to a dead-letter table. Never let dirty data reach Gold.

> **Run time:** ~5 min

## Architecture
```
Incoming Data
    │
    ▼
┌─────────────────────┐
│  Quality Rules       │  ← define rules as code
│  ─ NOT NULL checks  │
│  ─ Range validation │
│  ─ Enum validation  │
│  ─ FK integrity     │
│  ─ Duplicate check  │
└──────┬──────────────┘
       │
  ┌────┴─────┐
  ▼           ▼
PASS        FAIL
  │           │
Silver      dead_letter_transactions
(clean)     (quarantine + reason)
```

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Load the dirty data
df_dirty = spark.read.option('header','true').option('inferSchema','true') \
    .csv('Files/dirty_transactions.csv')

print(f'Incoming records: {df_dirty.count()}')
df_dirty.show(truncate=False)

## Step 1 — Define Quality Rules

In [ ]:
valid_txn_types = ['Deposit','Withdrawal','Transfer','Payment','Fee','Interest','Refund']
valid_channels  = ['Online','Mobile','ATM','Branch','Phone']
valid_statuses  = ['Completed','Pending','Failed']
max_amount      = 1_000_000
min_date        = '2015-01-01'
max_date        = '2026-12-31'

# Tag each record with all failing rules
df_checked = df_dirty \
    .withColumn('fail_null_txn_id',   F.col('TransactionID').isNull() | (F.col('TransactionID') == '')) \
    .withColumn('fail_null_account',  F.col('AccountID').isNull()     | (F.col('AccountID') == '')) \
    .withColumn('fail_null_customer', F.col('CustomerID').isNull()    | (F.col('CustomerID') == '')) \
    .withColumn('fail_amount_neg',    F.col('Amount') <= 0) \
    .withColumn('fail_amount_max',    F.col('Amount') > max_amount) \
    .withColumn('fail_date_format',   F.to_date(F.col('TransactionDate'), 'yyyy-MM-dd').isNull()) \
    .withColumn('fail_future_date',   F.to_date(F.col('TransactionDate'), 'yyyy-MM-dd') > F.lit(max_date)) \
    .withColumn('fail_txn_type',     ~F.col('TransactionType').isin(valid_txn_types)) \
    .withColumn('fail_channel',      ~F.col('Channel').isin(valid_channels)) \
    .withColumn('fail_status',       ~F.col('Status').isin(valid_statuses))

# Build human-readable failure reason
fail_cols = [c for c in df_checked.columns if c.startswith('fail_')]

reason_expr = F.concat_ws(' | ', *[
    F.when(F.col(c), F.lit(c.replace('fail_','')))
    for c in fail_cols
])

df_checked = df_checked \
    .withColumn('_failure_reasons', reason_expr) \
    .withColumn('_has_failures', F.lit(False))

for c in fail_cols:
    df_checked = df_checked.withColumn('_has_failures', F.col('_has_failures') | F.col(c))

df_checked = df_checked.drop(*fail_cols)

print(f'Total checked: {df_checked.count()}')
print(f'Passed:        {df_checked.filter(~F.col("_has_failures")).count()}')
print(f'Failed:        {df_checked.filter( F.col("_has_failures")).count()}')

## Step 2 — Route: Clean → Silver, Bad → Dead-Letter

In [ ]:
# PASS: clean records → silver
df_clean = df_checked.filter(~F.col('_has_failures')) \
    .drop('_has_failures','_failure_reasons') \
    .withColumn('_validated_at', F.current_timestamp())

df_clean.write.format('delta').mode('append').saveAsTable('silver_transactions_validated')
print(f'✅ Clean records written to silver_transactions_validated: {df_clean.count()}')

# FAIL: bad records → dead-letter table with reason
df_bad = df_checked.filter(F.col('_has_failures')) \
    .withColumn('_quarantined_at', F.current_timestamp()) \
    .withColumn('_source',         F.lit('dirty_transactions.csv'))

df_bad.write.format('delta').mode('append').saveAsTable('dead_letter_transactions')
print(f'⚠️  Bad records quarantined to dead_letter_transactions: {df_bad.count()}')

## Step 3 — Review Dead-Letter Table

In [ ]:
%%sql
SELECT TransactionID, AccountID, Amount, TransactionDate,
       TransactionType, Channel, Status,
       _failure_reasons
FROM dead_letter_transactions
ORDER BY _quarantined_at DESC

## Step 4 — Quality Summary Dashboard

In [ ]:
%%sql
-- Quality summary: which rules triggered most?
SELECT
    _failure_reasons AS FailureReason,
    COUNT(*)          AS Count
FROM dead_letter_transactions
GROUP BY _failure_reasons
ORDER BY Count DESC

## Step 5 — Quality Score KPI

In [ ]:
total   = df_checked.count()
passed  = df_clean.count()
failed  = df_bad.count()
score   = round(passed / total * 100, 1)

print('=' * 45)
print('  DATA QUALITY REPORT')
print('=' * 45)
print(f'  Total records:    {total}')
print(f'  Passed:           {passed}  ✅')
print(f'  Failed/Quarantined: {failed}  ⚠️')
print(f'  Quality Score:    {score}%')
print('  Target:           >= 95%')
status = '✅ PASS' if score >= 95 else '❌ FAIL — investigate dead-letter table'
print(f'  Result:           {status}')
print('=' * 45)